### 13. 구역별 이상 후보를 담은 `cal01_base_v3`

- 같은 구역의 센서 중 `평균 ±3σ` 밖인 값이 남으면 그 구역의 후보 `1`
  - 예를 들어 `Z4`는 `TC-Z4`, `OP-Z4`, `CP-Z4`를 확인함
  - `1`은 이상 후보이며 온도와 출력의 관계나 고장을 확정한 정답은 아님
- 검토한 공통 움직임은 해당 센서와 시간 범위에서만 후보 해제
  - 기록 끝 `2시간` 전체나 다른 센서까지 `0`으로 바꾸지 않음
- 계산 가능한 구역에 후보가 없으면 `0`
  - `0`도 정상 확정이 아님
- 기존 제외와 계산 불가능한 구역은 라벨을 비움
  - 제외와 보류 이유는 따로 저장함
- 원본 `v1`에서 직접 생성하며 `12번`이나 `v2` 실행은 필요 없음
  - 기존 행 라벨은 `LABEL_V1`으로 보존하고 새 행 전체 `LABEL`은 만들지 않음

### 1. 파일과 구역 설정

- `data/processed/cal01_base_v1.csv`와 `metadata/v1/dataset.json`을 읽음
  - 두 파일이 없으면 같은 폴더의 `05_interval_review.ipynb` 다음 `06_labeled_dataframe.ipynb`를 전체 실행함
- `SAVE_FILES=False`이면 계산 결과만 확인하고 파일은 저장하지 않음
- CSV와 생성 정보가 서로 맞지 않으면 계산을 멈춤
  - `06_labeled_dataframe.ipynb`에서 두 파일을 함께 다시 만듦


In [ ]:
import os
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from matplotlib.patches import Patch
from IPython.display import display

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

DATASET_VERSION = "v3"  # 생성할 데이터 버전
DATA_KIND = "raw"  # 행을 줄이지 않은 원래 관측 간격
SAVE_FILES = True  # False이면 CSV와 그림 저장 생략
ENC = "utf-8-sig"
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
INPUT_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v1.csv")
INPUT_METADATA = os.path.join(PROCESSED_DIR, "metadata", "v1", "dataset.json")
OUTPUT_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v3.csv")
REVIEW_DIR = os.path.join(PROCESSED_DIR, "review", DATASET_VERSION)
META_DIR = os.path.join(PROCESSED_DIR, "metadata", DATASET_VERSION)
FIG_DIR = os.path.join(ROOT, "figures", "namelesspark", "13_group_labeled_dataframe")
LABEL_VERSION = "group_lot_sigma_gt3_common_review_v3_20261021"
REVIEWED_SOURCE_SHA256 = "295a02af5018d8b66205156f54e942e3017ae371b193653dc96f0314ab8c48ab"

GROUPS = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}
SENSORS = [
    "CLN-IN", "TC-Z1", "TC-Z2", "OP-Z1", "OP-Z2",
    "TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6",
    "OP-Z3", "OP-Z4", "OP-Z5", "OP-Z6", "CP-Z4",
    "TC-Z7", "TC-Z8", "TC-Z9", "OP-Z7", "OP-Z8", "TC-OUT1", "TC-OUT2",
]
sensor_group = {}
for group, sensors in GROUPS.items():
    for sensor in sensors:
        sensor_group[sensor] = group

font_names = []
for font in font_manager.fontManager.ttflist:
    font_names.append(font.name)
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font_name in font_names:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False

base = pd.read_csv(INPUT_CSV, encoding=ENC, dtype=str, keep_default_na=False)
with open(INPUT_METADATA, encoding="utf-8") as file:
    input_metadata = json.load(file)
with open(INPUT_CSV, "rb") as file:
    source_sha256 = hashlib.file_digest(file, "sha256").hexdigest()
with open(INPUT_METADATA, "rb") as file:
    input_metadata_sha256 = hashlib.file_digest(file, "sha256").hexdigest()
if input_metadata.get("master_sha256") != source_sha256:
    raise ValueError("v1 CSV와 metadata/v1/dataset.json의 내용이 맞지 않음. 06_labeled_dataframe.ipynb로 같은 CSV와 설정을 다시 생성하세요.")
input_columns = input_metadata["input_columns"]
review_applied = source_sha256 == REVIEWED_SOURCE_SHA256

# 기존 행 라벨과 그 근거는 새 구역 라벨과 이름을 구분해서 보존함
legacy_rename = {"LABEL": "LABEL_V1"}
legacy_named = ["LABEL_STATUS", "LABEL_REASON", "LABEL_VERSION", "RULE", "ASSOCIATION_KIND", "MODEL_ELIGIBLE"]
for column in base.columns:
    if column in legacy_named or column.startswith("EVIDENCE_"):
        legacy_rename[column] = "V1_" + column
output = base.rename(columns=legacy_rename).copy()
output["DATA_KIND"] = DATA_KIND
output["DATASET_VERSION"] = DATASET_VERSION
times = pd.to_datetime(base["MEAS_DT"])
not_excluded = base["EXCLUDED"].eq("False")
values = base[SENSORS].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)

if SAVE_FILES:
    os.makedirs(REVIEW_DIR, exist_ok=True)
    os.makedirs(META_DIR, exist_ok=True)
    os.makedirs(FIG_DIR, exist_ok=True)
print("공통 검토표 적용:", review_applied)
print("원본 행:", len(base), "/ 기존 제외:", int((~not_excluded).sum()))

### 2. 같은 LOT의 `평균 ±3σ`

- 기존 제외 행을 빼고 같은 LOT의 센서별 평균과 표준편차를 계산함
  - 표준편차는 `ddof=0`
  - 기존 라벨이 `0`인지 `1`인지는 계산에 사용하지 않음
- `|현재값 - 평균| / 표준편차 > 3`일 때만 센서 후보
  - `3`과 같은 값은 후보가 아님
  - 결측이나 표준편차 `0`이면 해당 센서의 점수는 비움

In [ ]:
reference = values.loc[not_excluded]
lot_keys = base.loc[not_excluded, "LOT_NO"]
lot_mean = reference.groupby(lot_keys).mean()
lot_std = reference.groupby(lot_keys).std(ddof=0)
lot_count = reference.groupby(lot_keys).count()
scores = pd.DataFrame(np.nan, index=base.index, columns=SENSORS)
limit_records = []

for sensor in SENSORS:
    mean = base["LOT_NO"].map(lot_mean[sensor])
    std = base["LOT_NO"].map(lot_std[sensor]).replace(0, np.nan)
    scores[sensor] = values[sensor].subtract(mean).divide(std).abs().where(not_excluded)
    for lot in lot_mean.index:
        center = lot_mean.loc[lot, sensor]
        spread = lot_std.loc[lot, sensor]
        limit_records.append({
            "LOT_NO": lot,
            "SENSOR": sensor,
            "COUNT": int(lot_count.loc[lot, sensor]),
            "MEAN": center,
            "STD_DDOF0": spread,
            "LOWER": center - 3 * spread,
            "UPPER": center + 3 * spread,
        })

limits = pd.DataFrame(limit_records)
score_available = np.isfinite(scores)
stat_flags = scores.gt(3)

### 3. 확인한 공통 움직임만 후보에서 빼기

- `13개 LOT`의 `74개 센서별 시간 범위`만 사용함
  - `TC-Z3`, `OP-Z3`, `TC-Z6`, `OP-Z6`의 검토한 초과값만 해제
  - 시작은 포함하고 끝은 포함하지 않음
- 검토한 원본과 파일 내용 식별값이 같을 때만 적용함
  - 원본이 달라졌으면 아래 범위에 걸린 해당 구역을 보류함
  - 공통 움직임이라고 자동으로 `0`을 붙이지 않음

In [ ]:
common_ranges = pd.DataFrame([
    ['240001', 'TC-Z3', '2024-01-02 13:26:17', '2024-01-02 13:33:32', 'Z3 온도가 크게 올라간 봉우리'],
    ['240001', 'TC-Z3', '2024-01-02 13:36:45', '2024-01-02 13:40:40', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240001', 'OP-Z3', '2024-01-02 13:25:15', '2024-01-02 13:33:11', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240001', 'TC-Z6', '2024-01-02 13:35:23', '2024-01-02 13:40:04', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240001', 'TC-Z6', '2024-01-02 13:45:53', '2024-01-02 13:51:58', 'Z6 온도가 다시 올라간 부분'],
    ['240001', 'OP-Z6', '2024-01-02 13:33:33', '2024-01-02 13:33:51', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240001', 'OP-Z6', '2024-01-02 13:37:11', '2024-01-02 13:42:12', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240001', 'OP-Z6', '2024-01-02 13:47:30', '2024-01-02 13:51:52', 'Z6 온도가 다시 올라갈 때 출력이 낮아진 부분'],
    ['240024', 'TC-Z3', '2024-03-08 04:51:02', '2024-03-08 04:58:33', 'Z3 온도가 크게 올라간 봉우리'],
    ['240024', 'TC-Z3', '2024-03-08 05:06:56', '2024-03-08 05:12:00', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240024', 'OP-Z3', '2024-03-08 04:50:13', '2024-03-08 04:59:04', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240024', 'TC-Z6', '2024-03-08 05:00:36', '2024-03-08 05:07:40', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240024', 'TC-Z6', '2024-03-08 05:13:54', '2024-03-08 05:22:47', 'Z6 온도가 다시 올라간 부분'],
    ['240024', 'OP-Z6', '2024-03-08 04:59:07', '2024-03-08 05:12:02', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240037', 'TC-Z3', '2024-04-14 07:51:15', '2024-04-14 07:58:22', 'Z3 온도가 크게 올라간 봉우리'],
    ['240037', 'TC-Z3', '2024-04-14 08:02:53', '2024-04-14 08:06:37', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240037', 'OP-Z3', '2024-04-14 07:50:21', '2024-04-14 07:58:12', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240037', 'TC-Z6', '2024-04-14 08:00:27', '2024-04-14 08:07:41', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240037', 'TC-Z6', '2024-04-14 08:11:08', '2024-04-14 08:17:53', 'Z6 온도가 다시 올라간 부분'],
    ['240037', 'OP-Z6', '2024-04-14 08:00:22', '2024-04-14 08:08:56', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240038', 'TC-Z3', '2024-04-16 12:06:47', '2024-04-16 12:08:07', 'Z3 온도가 크게 올라간 봉우리'],
    ['240038', 'TC-Z3', '2024-04-16 12:16:04', '2024-04-16 12:19:08', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240038', 'OP-Z3', '2024-04-16 12:04:01', '2024-04-16 12:10:40', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240038', 'TC-Z6', '2024-04-16 12:24:50', '2024-04-16 12:28:31', 'Z6 온도가 다시 올라간 부분'],
    ['240038', 'OP-Z6', '2024-04-16 12:16:49', '2024-04-16 12:19:22', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240058', 'TC-Z3', '2024-06-09 07:26:03', '2024-06-09 07:32:03', 'Z3 온도가 크게 올라간 봉우리'],
    ['240058', 'OP-Z3', '2024-06-09 07:24:57', '2024-06-09 07:31:37', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240058', 'TC-Z6', '2024-06-09 07:37:05', '2024-06-09 07:40:38', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240058', 'OP-Z6', '2024-06-09 07:36:19', '2024-06-09 07:44:52', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240061', 'TC-Z3', '2024-06-15 03:14:01', '2024-06-15 03:18:01', 'Z3 온도가 크게 올라간 봉우리'],
    ['240061', 'TC-Z6', '2024-06-15 03:26:03', '2024-06-15 03:26:13', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240061', 'OP-Z6', '2024-06-15 03:24:16', '2024-06-15 03:28:48', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240069', 'TC-Z3', '2024-07-08 04:51:27', '2024-07-08 04:57:43', 'Z3 온도가 크게 올라간 봉우리'],
    ['240069', 'OP-Z3', '2024-07-08 04:50:27', '2024-07-08 04:57:04', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240069', 'TC-Z6', '2024-07-08 05:03:14', '2024-07-08 05:07:14', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240069', 'TC-Z6', '2024-07-08 05:10:41', '2024-07-08 05:17:22', 'Z6 온도가 다시 올라간 부분'],
    ['240069', 'OP-Z6', '2024-07-08 04:59:56', '2024-07-08 05:09:12', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240091', 'TC-Z3', '2024-09-09 07:07:43', '2024-09-09 07:15:04', 'Z3 온도가 크게 올라간 봉우리'],
    ['240091', 'TC-Z3', '2024-09-09 07:23:56', '2024-09-09 07:29:22', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240091', 'OP-Z3', '2024-09-09 07:06:43', '2024-09-09 07:15:17', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240091', 'TC-Z6', '2024-09-09 07:17:22', '2024-09-09 07:22:58', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240091', 'TC-Z6', '2024-09-09 07:30:32', '2024-09-09 07:39:25', 'Z6 온도가 다시 올라간 부분'],
    ['240091', 'OP-Z6', '2024-09-09 07:14:54', '2024-09-09 07:26:08', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240091', 'OP-Z6', '2024-09-09 07:31:15', '2024-09-09 07:39:52', 'Z6 온도가 다시 올라갈 때 출력이 낮아진 부분'],
    ['240108', 'TC-Z3', '2024-10-18 10:14:29', '2024-10-18 10:17:41', 'Z3 온도가 크게 올라간 봉우리'],
    ['240108', 'TC-Z3', '2024-10-18 10:24:39', '2024-10-18 10:27:22', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240108', 'OP-Z3', '2024-10-18 10:12:36', '2024-10-18 10:18:21', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240108', 'TC-Z6', '2024-10-18 10:22:28', '2024-10-18 10:27:25', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240108', 'TC-Z6', '2024-10-18 10:31:59', '2024-10-18 10:37:52', 'Z6 온도가 다시 올라간 부분'],
    ['240108', 'OP-Z6', '2024-10-18 10:21:08', '2024-10-18 10:24:24', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240108', 'OP-Z6', '2024-10-18 10:24:25', '2024-10-18 10:28:52', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240114', 'TC-Z3', '2024-11-04 19:15:40', '2024-11-04 19:22:12', 'Z3 온도가 크게 올라간 봉우리'],
    ['240114', 'TC-Z3', '2024-11-04 19:27:28', '2024-11-04 19:30:03', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240114', 'OP-Z3', '2024-11-04 19:14:08', '2024-11-04 19:22:40', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240114', 'TC-Z6', '2024-11-04 19:26:25', '2024-11-04 19:30:42', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240114', 'TC-Z6', '2024-11-04 19:34:28', '2024-11-04 19:41:02', 'Z6 온도가 다시 올라간 부분'],
    ['240114', 'OP-Z6', '2024-11-04 19:23:48', '2024-11-04 19:32:19', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240114', 'OP-Z6', '2024-11-04 19:32:24', '2024-11-04 19:32:32', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240117', 'TC-Z3', '2024-11-13 12:56:18', '2024-11-13 13:02:20', 'Z3 온도가 크게 올라간 봉우리'],
    ['240117', 'OP-Z3', '2024-11-13 12:55:18', '2024-11-13 13:01:48', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240117', 'TC-Z6', '2024-11-13 13:07:56', '2024-11-13 13:10:48', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240117', 'TC-Z6', '2024-11-13 13:19:26', '2024-11-13 13:26:41', 'Z6 온도가 다시 올라간 부분'],
    ['240117', 'OP-Z6', '2024-11-13 13:04:57', '2024-11-13 13:10:14', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240117', 'OP-Z6', '2024-11-13 13:10:15', '2024-11-13 13:13:25', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240121', 'TC-Z3', '2024-11-24 16:39:09', '2024-11-24 16:46:55', 'Z3 온도가 크게 올라간 봉우리'],
    ['240121', 'TC-Z3', '2024-11-24 16:51:56', '2024-11-24 16:56:59', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240121', 'OP-Z3', '2024-11-24 16:38:07', '2024-11-24 16:46:55', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240121', 'TC-Z6', '2024-11-24 16:48:36', '2024-11-24 16:55:42', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240121', 'TC-Z6', '2024-11-24 16:59:16', '2024-11-24 17:07:13', 'Z6 온도가 다시 올라간 부분'],
    ['240121', 'OP-Z6', '2024-11-24 16:49:30', '2024-11-24 16:58:00', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240133', 'TC-Z3', '2024-12-28 15:08:43', '2024-12-28 15:11:51', 'Z3 온도가 크게 올라간 봉우리'],
    ['240133', 'OP-Z3', '2024-12-28 15:06:24', '2024-12-28 15:13:15', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240133', 'TC-Z6', '2024-12-28 15:28:37', '2024-12-28 15:31:53', 'Z6 온도가 다시 올라간 부분'],
    ['240133', 'OP-Z6', '2024-12-28 15:16:31', '2024-12-28 15:22:41', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
], columns=["LOT_NO", "SENSOR", "START", "END", "REASON"])
common_ranges["START"] = pd.to_datetime(common_ranges["START"])
common_ranges["END"] = pd.to_datetime(common_ranges["END"])
common_ranges.insert(0, "REVIEW_ID", ["C" + str(i).zfill(3) for i in range(1, len(common_ranges) + 1)])

removed_flags = pd.DataFrame(False, index=base.index, columns=SENSORS)
unverified_common = pd.DataFrame(False, index=base.index, columns=SENSORS)
removed_review_ids = pd.DataFrame("", index=base.index, columns=SENSORS)
range_counts = []
range_status = []

for row in common_ranges.itertuples(index=False):
    in_range = base["LOT_NO"].eq(row.LOT_NO) & times.ge(row.START) & times.lt(row.END) & not_excluded
    hit = in_range & stat_flags[row.SENSOR]
    if review_applied:
        removed_flags.loc[hit, row.SENSOR] = True
        removed_review_ids.loc[hit, row.SENSOR] += row.REVIEW_ID + ";"
        range_counts.append(int(hit.sum()))
        range_status.append("검토한 공통 움직임 해제")
    else:
        unverified_common.loc[in_range, row.SENSOR] = True
        range_counts.append(0)
        range_status.append("원본 변경으로 미적용, 해당 구역 보류")

common_ranges["REMOVED_SENSOR_ROWS"] = range_counts
common_ranges["APPLY_STATUS"] = range_status
candidate_flags = stat_flags & ~removed_flags
print("공통 움직임으로 후보에서 뺀 센서값:", int(removed_flags.to_numpy().sum()))

### 4. 구역별 `0`, `1`, 보류, 제외

- 해당 구역의 센서 점수를 모두 계산할 수 있고 후보가 남으면 `1`
  - 다른 구역의 후보는 이 구역의 라벨을 바꾸지 않음
- 모두 계산할 수 있고 남은 후보가 없으면 `0`
  - 공통 움직임을 해제한 뒤 다른 센서 후보가 남으면 `1` 유지
- 해당 구역의 점수가 부족하거나 공통 검토표를 적용할 수 없으면 보류
- 기존 `EXCLUDED=True`는 제외
  - `058` 정기수리 정지와 `061` 재가동에 대한 팀 판단도 유지

In [ ]:
label_columns = {}
group_input_columns = {}
group_valid = {}
group_label_tables = []
group_counts = []
lot_group_counts = []
common_row_tables = []

for group, sensors in GROUPS.items():
    label_col = group + "__LABEL"
    status_col = group + "__LABEL_STATUS"
    reason_col = group + "__LABEL_REASON"
    sensor_col = group + "__LABEL_SENSORS"
    review_col = group + "__COMMON_REVIEW_IDS"
    label_columns[group] = label_col

    available = score_available[sensors].all(axis=1) & not_excluded
    waiting_common = unverified_common[sensors].any(axis=1) & not_excluded
    usable = available & ~waiting_common
    group_valid[group] = usable
    before_review = stat_flags[sensors].any(axis=1)
    remaining = candidate_flags[sensors].any(axis=1)
    common_here = removed_flags[sensors].any(axis=1)

    labels = pd.Series(pd.NA, index=base.index, dtype="Int64")
    labels.loc[usable] = remaining.loc[usable].astype("int64")
    status = pd.Series("보류", index=base.index)
    status.loc[usable & ~remaining] = "후보 아님"
    status.loc[usable & remaining] = "후보"
    status.loc[~not_excluded] = "제외"
    reasons = pd.Series("해당 구역의 센서 결측 또는 점수 계산 불가", index=base.index)
    reasons.loc[usable & ~remaining] = "이 구역에 남아 있는 평균±3σ 초과 후보 없음"
    reasons.loc[usable & remaining] = "이 구역에 같은 LOT의 평균±3σ 초과 후보가 남음"
    reasons.loc[usable & common_here & ~remaining] = "검토한 공통 움직임을 뺀 뒤 이 구역의 후보가 남지 않음"
    reasons.loc[usable & common_here & remaining] = "공통 움직임을 뺐지만 같은 구역의 다른 센서 후보가 남음"
    reasons.loc[waiting_common] = "원본이 바뀌어 공통 검토표를 적용하지 않음, 다시 검토 필요"
    reasons.loc[~not_excluded] = base.loc[~not_excluded, "EXCLUSION_REASON"]
    reasons.loc[~not_excluded & base["LOT_NO"].eq("240058")] = "정기수리를 위해 온도를 내리는 구간, 팀 확정과 기존 제외 유지"
    reasons.loc[~not_excluded & base["LOT_NO"].eq("240061")] = "정기수리 후 재가동하며 온도를 올리는 구간, 팀 확정과 기존 제외 유지"

    evidence = pd.Series("", index=base.index)
    review_ids = pd.Series("", index=base.index)
    for sensor in sensors:
        evidence.loc[labels.eq(1).fillna(False) & candidate_flags[sensor]] += sensor + ";"
        review_ids += removed_review_ids[sensor]
    label_table = pd.DataFrame({
        label_col: labels,
        status_col: status,
        reason_col: reasons,
        sensor_col: evidence.str.rstrip(";"),
        review_col: review_ids.str.rstrip(";"),
    })
    group_label_tables.append(label_table)

    # 기존 v1 입력 중 이 구역에 해당하는 값만 제안 목록에 남김
    allowed = []
    for column in input_columns:
        related = column in sensors
        for sensor in sensors:
            related = related or column.startswith(sensor + "_")
        if group not in ["CLN", "Z1", "Z2"] and column.startswith("GRADE_"):
            related = True
        if related:
            allowed.append(column)
    group_input_columns[group] = allowed
    group_counts.append({
        "GROUP": group,
        "SENSORS": ", ".join(sensors),
        "LABEL0": int(labels.eq(0).sum()),
        "LABEL1": int(labels.eq(1).sum()),
        "PENDING": int(status.eq("보류").sum()),
        "EXCLUDED": int(status.eq("제외").sum()),
        "COMMON_TO0": int((usable & before_review & labels.eq(0).fillna(False)).sum()),
    })
    for lot, block in base.groupby("LOT_NO", sort=False):
        index = block.index
        lot_group_counts.append({
            "LOT_NO": lot,
            "GRADE": block["GRADE"].iloc[0],
            "GROUP": group,
            "LABEL0": int(labels.loc[index].eq(0).sum()),
            "LABEL1": int(labels.loc[index].eq(1).sum()),
            "PENDING": int(status.loc[index].eq("보류").sum()),
            "EXCLUDED": int(status.loc[index].eq("제외").sum()),
        })
    touched = common_here | waiting_common
    common_rows = base.loc[touched, ["ROW_ID", "LOT_NO", "MEAS_DT"]].copy()
    common_rows["GROUP"] = group
    common_rows["LABEL"] = labels.loc[touched]
    common_rows["STATUS"] = status.loc[touched]
    common_rows["REVIEW_IDS"] = review_ids.loc[touched].str.rstrip(";")
    common_row_tables.append(common_rows)

output = pd.concat([output] + group_label_tables, axis=1)
output["GROUP_LABEL_VERSION"] = LABEL_VERSION
group_counts = pd.DataFrame(group_counts)
lot_group_counts = pd.DataFrame(lot_group_counts)
common_rows = pd.concat(common_row_tables, ignore_index=True)
display(group_counts.rename(columns={
    "GROUP": "구역", "SENSORS": "센서", "LABEL0": "후보 아님 0",
    "LABEL1": "이상 후보 1", "PENDING": "보류", "EXCLUDED": "기존 제외",
    "COMMON_TO0": "공통 움직임을 빼서 0",
}))
print("LOT별 구역 후보 1 개수")
display(lot_group_counts.pivot(index="LOT_NO", columns="GROUP", values="LABEL1"))

### 5. 표의 시간 범위를 노란 배경으로 보기

- 각 표 바로 아래에 같은 LOT의 센서 그래프를 표시함
  - 주황 점은 남은 센서 후보
  - 회색 점은 공통 움직임으로 해제한 센서값
  - 회색 배경은 기존 제외
- 기존 `8개 검토 사례`와 `058`, `061`의 팀 판단을 함께 봄
  - 같은 LOT의 사례는 한 그림에 묶음
  - 노란 배경 전체를 후보 `1`로 바꾸지 않음
- 정비 기록은 설명을 돕는 자료이며 라벨을 바꾸는 조건으로 쓰지 않음

In [ ]:
review_cases = [
    {
        "lot": "240024", "sensor": "CP-Z4",
        "start": "2024-03-08 04:55", "end": "2024-03-08 05:03",
        "plot": ["CP-Z4", "TC-Z4"],
        "reason": "CP가 0.45 부근에서 0.383640까지 내려갔다 회복함. 같은 강종 108의 최저 0.404613보다 낮아 다시 살펴볼 이유가 있음.",
        "record": "03/10 15:10 정기점검 이상없음 감지 기록이 있음(WO-2024-00378). 이 기록을 라벨 조건으로 쓰지는 않음.",
    },
    {
        "lot": "240037", "sensor": "CP-Z4",
        "start": "2024-04-13 23:57", "end": "2024-04-14 00:00",
        "plot": ["CP-Z4", "TC-Z4"],
        "reason": "온도 변화는 작은데 CP는 0.606760까지 올라감. 같은 강종 121, 133의 최고 0.466862보다 큰 급등임.",
        "record": "강종 변경 41분 뒤임. 04/14 15:04 확인 중 감지 기록과 실제 작업일 04/23은 구분함(WO-2024-00627).",
    },
    {
        "lot": "240037", "sensor": "CP-Z4",
        "start": "2024-04-14 03:10", "end": "2024-04-14 03:12",
        "plot": ["CP-Z4", "TC-Z4"],
        "reason": "CP가 0.522137까지 짧게 다시 올라감. 앞의 23:57 급등과 약 3시간 떨어진 별도 변화임.",
        "record": "같은 날 15:04 확인 중 감지 기록이 있음(WO-2024-00627). 두 급등 사이를 이어서 1로 붙이지 않음.",
    },
    {
        "lot": "240069", "sensor": "CP-Z4",
        "start": "2024-07-08 02:30", "end": "2024-07-08 02:40",
        "plot": ["CP-Z4", "TC-Z4", "OP-Z4"],
        "reason": "CP가 약 0.45에서 0.909111까지 크게 올라갔다 복귀함. 1분 평균 최고도 0.883902로, 같은 강종 044, 061, 091의 최고 0.478683보다 큼.",
        "record": "08:49 절연 저하 감지, 11:11 작업 시작, O링 3개 예방 교체 기록이 있음(WO-2024-01094). CP 급등과의 연결은 별도로 살펴볼 내용임.",
    },
    {
        "lot": "240108", "sensor": "TC-Z4",
        "start": "2024-10-18 05:50", "end": "2024-10-18 06:20",
        "plot": ["TC-Z4", "OP-Z4"],
        "reason": "1분 평균으로 본 30분 온도 변화 폭은 2.70℃임. 같은 강종 024의 최대 30분 변화 폭 1.42℃보다 큼. 온도와 출력이 반대로 움직이는 사실 자체를 이상 근거로 삼지는 않음.",
        "record": "15:50 원인 미상 감지 기록이 있음. 실제 작업은 10/26 이후임(WO-2024-01678).",
    },
    {
        "lot": "240117", "sensor": "CP-Z4",
        "start": "2024-11-13 13:02", "end": "2024-11-13 13:10",
        "plot": ["CP-Z4", "TC-Z4"],
        "reason": "CP가 약 0.45에서 0.367097까지 내려갔다 회복함. 09:00 이전 승온 제외 구간과는 다른 변화임.",
        "record": "마모량 초과 작업은 12:44에 끝남(WO-2024-01854). 큰 하강은 13:02:50부터라 종료 18분 50초 뒤임. 같은 강종 비교 LOT는 없음.",
    },
    {
        "lot": "240121", "sensor": "CP-Z4",
        "start": "2024-11-24 14:26:05", "end": "2024-11-24 14:27:09",
        "plot": ["CP-Z4", "TC-Z4"],
        "reason": "CP가 0.45 부근에서 0.422714까지 짧게 내려감. 아래 16시대 Z6 반등과 시간과 센서가 다르므로 별도로 판단함.",
        "record": "다음 날 14:19 설비 트립 감지, 11/30 작업 기록이 있음(WO-2024-01922). 라벨은 이 시간 연관이 아니라 ±3σ로 계산함.",
    },
    {
        "lot": "240121", "sensor": "TC-Z6",
        "start": "2024-11-24 16:56:02", "end": "2024-11-24 17:01:47",
        "plot": ["TC-Z6", "OP-Z6", "CP-Z4"],
        "reason": "Z3 고점 16:42, Z6 저점 16:53 뒤의 반등임. 037, 133에도 비슷한 반등이 있어 검토한 TC-Z6 초과값은 공통 움직임으로 후보에서 뺌.",
        "record": "같은 시간의 CP 후보는 Z4에 따로 남김. Z6의 공통 움직임 판정으로 다른 구역의 라벨을 바꾸지 않음.",
    },
]

# 기존 여덟 사례를 보존하고 팀이 확정한 정지와 재가동을 함께 표시함
team_cases = [
    {
        "lot": "240058",
        "sensor": "TC-Z3",
        "start": "2024-06-09 08:00:00",
        "end": "2024-06-09 09:02:54",
        "plot": ["TC-Z3", "OP-Z3"],
        "reason": "정기수리를 위해 소둔로 온도를 내리는 구간으로 팀이 확정했음",
        "record": "기존 EXCLUDED와 빈 구역 라벨을 유지함",
    },
    {
        "lot": "240061",
        "sensor": "TC-Z3",
        "start": "2024-06-15 01:04:00",
        "end": "2024-06-15 02:30:00",
        "plot": ["TC-Z3", "OP-Z3"],
        "reason": "정기수리 후 다시 가동하며 온도를 올리는 구간으로 팀이 확정했음",
        "record": "기존 EXCLUDED와 빈 구역 라벨을 유지함",
    },
]
plot_cases = [
    review_cases[0:1],
    review_cases[1:3],
    review_cases[3:4],
    review_cases[4:5],
    review_cases[5:6],
    review_cases[6:8],
    team_cases[0:1],
    team_cases[1:2],
]
domain_rows = []
domain_counts = []
for figure_number, cases in enumerate(plot_cases, 1):
    lot = cases[0]["lot"]
    case_table = []
    channels = []
    starts = []
    ends = []
    for case in cases:
        group = sensor_group[case["sensor"]]
        start = pd.Timestamp(case["start"])
        end = pd.Timestamp(case["end"])
        starts.append(start)
        ends.append(end)
        selected = base["LOT_NO"].eq(lot) & times.ge(start) & times.lt(end)
        labels = output.loc[selected, group + "__LABEL"]
        status = output.loc[selected, group + "__LABEL_STATUS"]
        record = {
            "LOT": lot,
            "구역": group,
            "시작": start,
            "끝 미포함": end,
            "후보 1": int(labels.eq(1).sum()),
            "후보 아님 0": int(labels.eq(0).sum()),
            "보류": int(status.eq("보류").sum()),
            "기존 제외": int(status.eq("제외").sum()),
        }
        case_table.append(record)
        saved_record = dict(record)
        saved_record["설명"] = case["reason"]
        saved_record["관련 기록"] = case["record"]
        domain_counts.append(saved_record)
        detail = output.loc[selected, ["ROW_ID", "LOT_NO", "MEAS_DT", group + "__LABEL", group + "__LABEL_STATUS"]].copy()
        detail["GROUP"] = group
        detail["CASE_START"] = start
        detail["CASE_END"] = end
        domain_rows.append(detail)
        for channel in case["plot"]:
            if channel not in channels:
                channels.append(channel)

    display(pd.DataFrame(case_table))
    view = base["LOT_NO"].eq(lot) & times.ge(min(starts) - pd.Timedelta(minutes=5)) & times.lt(max(ends) + pd.Timedelta(minutes=5))
    index = times.loc[view].sort_values().index
    gaps = times.loc[index].diff().dt.total_seconds()
    segments = (gaps.gt(15) | gaps.le(0)).cumsum()
    figure, axes = plt.subplots(len(channels), 1, figsize=(13, 2.5 * len(channels)), sharex=True, layout="constrained", squeeze=False)
    for axis, channel in zip(axes[:, 0], channels):
        for start, end in zip(starts, ends):
            axis.axvspan(start, end, color="#f9db73", alpha=0.25)
        for segment, piece in base.loc[index].groupby(segments, sort=False):
            axis.plot(times.loc[piece.index], values.loc[piece.index, channel], color="#355f82", linewidth=0.8, label="센서 원자료" if segment == segments.iloc[0] else None)
        excluded_blocks = (not_excluded.loc[index].ne(not_excluded.loc[index].shift()) | segments.ne(segments.shift())).cumsum()
        for segment, piece in base.loc[index].groupby(excluded_blocks, sort=False):
            if not not_excluded.loc[piece.index[0]]:
                axis.axvspan(times.loc[piece.index[0]], times.loc[piece.index[-1]] + pd.Timedelta(seconds=1), color="#aeb5bd", alpha=0.45)
        removed = removed_flags.loc[index, channel]
        channel_group = sensor_group[channel]
        hit = candidate_flags.loc[index, channel] & output.loc[index, channel_group + "__LABEL"].eq(1).fillna(False)
        axis.scatter(times.loc[index[removed]], values.loc[index[removed], channel], s=10, color="#83878d", label="회색 점: 공통 움직임 해제", zorder=3)
        axis.scatter(times.loc[index[hit]], values.loc[index[hit], channel], s=10, color="#df650b", label="주황 점: 남은 센서 후보", zorder=4)
        mean = lot_mean.loc[lot, channel]
        std = lot_std.loc[lot, channel]
        axis.axhline(mean - 3 * std, color="#747d87", linestyle="--", linewidth=0.8)
        axis.axhline(mean + 3 * std, color="#747d87", linestyle="--", linewidth=0.8)
        unit = "단위 미확인" if channel.startswith("CP") else ("%" if channel.startswith("OP") else "℃")
        axis.set_ylabel(channel + "\n" + unit)
        axis.grid(alpha=0.15)
    handles, legend_labels = axes[0, 0].get_legend_handles_labels()
    if (~not_excluded.loc[index]).any():
        handles.append(Patch(facecolor="#aeb5bd", alpha=0.45))
        legend_labels.append("회색 배경: 팀이 학습에서 제외한 구간")
    axes[0, 0].legend(handles, legend_labels, loc="lower left", bbox_to_anchor=(0, 1.02), ncol=2, fontsize=8)
    axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    axes[-1, 0].set_xlabel("2024년 실제 날짜와 시각")
    figure.suptitle(lot + " | 노란 배경은 위 표의 시간 범위, 점선은 평균 ±3σ", fontsize=12)
    if SAVE_FILES:
        figure.savefig(os.path.join(FIG_DIR, str(figure_number).zfill(2) + "_" + lot + "_group_labels.png"), dpi=140)
    plt.show()
    plt.close(figure)
    for case in cases:
        print(case["reason"])
        print(case["record"])

domain_counts = pd.DataFrame(domain_counts)
domain_rows = pd.concat(domain_rows, ignore_index=True)

### 6. 중앙 `60%` 대비 온도 차이와 지속 시간

- 관측 시간의 앞뒤 `20%`를 뺀 가운데 `60%`의 센서별 중앙값을 사용함
  - 기존 제외와 해당 센서 결측을 빼며 기존 라벨은 조건으로 쓰지 않음
- `TC-Z3`부터 `TC-Z6`까지 온도 차이, `5℃` 초과 상승과 하강의 지속 시간, 최근 `5분` 초과 누적을 남김
  - 같은 계산을 사용한 기존 `v2`의 온도 특징과 의미가 같음
  - 이 값으로 구역 라벨을 바꾸지 않음
- LOT가 끝난 뒤 계산하는 사후 특징
  - 실제 설정온도나 강판 온도가 아님
  - `5℃`, `5분`은 품질 불량 확정 기준이 아님
  - 미래 시점 예측에 사용할 학습 입력 목록에서는 제외함

In [ ]:
THERMAL_SENSORS = ["TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6"]
CENTRAL_START_FRACTION = 0.2
CENTRAL_END_FRACTION = 0.8
TEMP_LIMIT_C = 5.0
ACCUMULATION_SECONDS = 300
MAX_GAP_SECONDS = 15
thermal_suffixes = [
    "_중앙값대비온도차",
    "_5도초과상승_지속초",
    "_5도초과하강_지속초",
    "_최근5분_5도초과누적",
    "_중앙값대비_표준화편차",
]
thermal_input_columns = []
coverage_columns = []
for sensor in THERMAL_SENSORS:
    thermal_input_columns += [sensor + suffix for suffix in thermal_suffixes]
    coverage_columns.append(sensor + "_최근5분_확보초")

thermal_features = pd.DataFrame(np.nan, index=base.index, columns=thermal_input_columns + coverage_columns)
thermal_reference_records = []
thermal_summary_records = []

for lot, lot_rows in base.groupby("LOT_NO", sort=False):
    index = times.loc[lot_rows.index].sort_values().index
    clock = times.loc[index]
    start, end = clock.iloc[0], clock.iloc[-1]
    ref_start = start + (end - start) * CENTRAL_START_FRACTION
    ref_end = start + (end - start) * CENTRAL_END_FRACTION
    central = clock.ge(ref_start) & clock.le(ref_end) & not_excluded.loc[index]
    step_seconds = clock.diff().dt.total_seconds()

    for sensor in THERMAL_SENSORS:
        reference_values = values.loc[index[central], sensor].dropna()
        center = reference_values.median()
        spread = reference_values.std(ddof=0)
        thermal_reference_records.append({
            "LOT_NO": lot, "SENSOR": sensor, "LOT_START": start, "LOT_END": end,
            "REFERENCE_START_INCLUDED": ref_start, "REFERENCE_END_INCLUDED": ref_end,
            "COUNT": len(reference_values), "MEDIAN_C": center, "STD_DDOF0_C": spread,
        })
        temperature = values.loc[index, sensor]
        good = not_excluded.loc[index] & temperature.notna() & np.isfinite(center)
        deviation = temperature.subtract(center).where(good)
        connected = good & good.shift(fill_value=False) & step_seconds.gt(0) & step_seconds.le(MAX_GAP_SECONDS)
        above = deviation.gt(TEMP_LIMIT_C)
        below = deviation.lt(-TEMP_LIMIT_C)
        up_start = clock.where(above & ~(above.shift(fill_value=False) & connected)).ffill()
        down_start = clock.where(below & ~(below.shift(fill_value=False) & connected)).ffill()
        thermal_features.loc[index, sensor + "_중앙값대비온도차"] = deviation
        thermal_features.loc[index, sensor + "_5도초과상승_지속초"] = clock.subtract(up_start).dt.total_seconds().where(above, 0).where(good)
        thermal_features.loc[index, sensor + "_5도초과하강_지속초"] = clock.subtract(down_start).dt.total_seconds().where(below, 0).where(good)
        if spread > 0:
            thermal_features.loc[index, sensor + "_중앙값대비_표준화편차"] = deviation / spread

        segments = (~connected).cumsum()
        for segment, segment_rows in lot_rows.loc[index[good]].groupby(segments.loc[index[good]], sort=False):
            segment_index = times.loc[segment_rows.index].sort_values().index
            segment_times = times.loc[segment_index]
            seconds = segment_times.subtract(segment_times.iloc[0]).dt.total_seconds().to_numpy()
            excess = deviation.loc[segment_index].abs().subtract(TEMP_LIMIT_C).clip(lower=0).to_numpy()
            area = np.r_[0.0, np.cumsum(excess[:-1] * np.diff(seconds))]
            recent_area = area - np.interp(seconds - ACCUMULATION_SECONDS, seconds, area)
            coverage = np.minimum(seconds, ACCUMULATION_SECONDS)
            thermal_features.loc[segment_index, sensor + "_최근5분_확보초"] = coverage
            thermal_features.loc[segment_index, sensor + "_최근5분_5도초과누적"] = np.where(
                coverage >= ACCUMULATION_SECONDS, np.maximum(recent_area, 0), np.nan
            )

        thermal_summary_records.append({
            "LOT": lot, "센서": sensor, "기준 중앙값 ℃": center,
            "최대 절대 온도차 ℃": deviation.abs().max(),
            "최장 상승 지속초": thermal_features.loc[index, sensor + "_5도초과상승_지속초"].max(),
            "최장 하강 지속초": thermal_features.loc[index, sensor + "_5도초과하강_지속초"].max(),
            "최대 5분 초과누적 ℃×초": thermal_features.loc[index, sensor + "_최근5분_5도초과누적"].max(),
        })

thermal_reference = pd.DataFrame(thermal_reference_records)
thermal_summary = pd.DataFrame(thermal_summary_records)
thermal_complete = np.isfinite(thermal_features[thermal_input_columns]).all(axis=1)
thermal_features["THERMAL_FEATURES_COMPLETE"] = thermal_complete.map({True: "True", False: "False"})

output = pd.concat([output, thermal_features], axis=1)
display(thermal_summary)
print("사후 온도 특징 20개를 모두 계산할 수 있는 행:", int(thermal_complete.sum()))

### 7. 새 파일과 생성 기준 저장

- 원본 센서값, 기존 파생값, 행 순서와 기존 제외는 유지함
- 기존 행 라벨 근거는 `LABEL_V1`, `V1_LABEL_REASON`, `V1_RULE`, `V1_EVIDENCE_*`로 구분함
  - 그 외 기존 문맥 열도 새 라벨의 근거로 사용하지 않음
- `GROUP__LABEL`이 비교할 구역 라벨
  - 라벨과 근거 열은 모델 입력으로 사용하지 않음
  - 구역 라벨은 통계 규칙과 공통 움직임 검토로 만든 후보이며 고장 정답이 아님
- 같은 Z-score 규칙으로 만든 라벨을 같은 규칙으로 채점하면 독립 평가가 아님
  - `117`, `121`, `133`도 공통 움직임 검토에 사용한 자료임
  - 미래 고장 성능과 구역 후보 규칙의 재현 정도를 구분해서 해석해야 함
- `Z9`, 냉각처럼 이 규칙에서 후보가 없는 구역은 모델 평가에서도 표시해야 함
  - 같은 LOT의 평균 자체가 달라진 경우는 이 규칙만으로 찾기 어려움
- CSV 내용에 맞는 생성 정보도 `metadata/v3/`에 따로 남김
  - 나중에 다른 CSV를 만들어도 보관한 CSV의 생성 정보를 다시 찾을 수 있음

- 이전 생성 정보는 `history_...json`에도 남김.
  - 같은 CSV가 만들어져도 생성 근거가 달라지면 이전 기록을 덮지 않음.


In [ ]:
legacy_context_columns = [
    "ANY_SENSOR_CANDIDATE", "CONTROL_RESPONSE_CONTEXT", "PERIODIC_CONTEXT",
    "LOT_BOUNDARY_CONTEXT", "MAINT_WORK_OVERLAP", "SCHEDULED_REPAIR_CONTEXT",
    "REPAIR_PREPOST_24H_CONTEXT", "GRADE_CHANGE_NEARBY_30MIN",
    "CP_RETURN_TO_BASELINE_CONTEXT", "LOT_CONTEXT_ID", "SENSOR_CANDIDATE_IDS",
]
label_detail_columns = []
for group in GROUPS:
    label_detail_columns += [
        group + "__LABEL_STATUS",
        group + "__LABEL_REASON",
        group + "__LABEL_SENSORS",
        group + "__COMMON_REVIEW_IDS",
    ]
forbidden = []
for column in output.columns:
    if column not in input_columns:
        forbidden.append(column)

manifest = {
    "dataset_version": DATASET_VERSION,
    "data_kind": DATA_KIND,
    "master_file": os.path.basename(OUTPUT_CSV),
    "rows": len(output),
    "columns": len(output.columns),
    "row_key": ["ROW_ID", "LOT_NO", "MEAS_DT"],
    "groups": GROUPS,
    "label_columns": label_columns,
    "label_dtype": "nullable Int64; CSV blank means excluded or pending, read status column",
    "group_input_columns": group_input_columns,
    "input_columns": input_columns,
    "raw_sensor_columns": SENSORS,
    "model_input_forbidden": forbidden,
    "metadata_groups": {
        "row_keys": ["ROW_ID", "LOT_NO", "MEAS_DT"],
        "raw_sensors": SENSORS,
        "existing_input_columns": input_columns,
        "legacy_label_and_evidence": list(legacy_rename.values()),
        "legacy_context": legacy_context_columns,
        "labels": list(label_columns.values()),
        "label_details": label_detail_columns,
        "retrospective_features": thermal_input_columns,
        "retrospective_coverage": coverage_columns,
    },
    "legacy_rename": legacy_rename,
    "legacy_meaning": "v1 label and evidence values are preserved; they do not determine new group labels",
    "group_counts": group_counts.to_dict("records"),
    "label_policy": {
        "version": LABEL_VERSION,
        "reference": "same LOT, EXCLUDED=False, sensor-specific available values, no label filter",
        "std_ddof": 0,
        "threshold": 3,
        "operator": ">",
        "one": "all group sensor scores available and at least one >3sigma candidate remains after reviewed sensor-specific removals",
        "zero": "all group scores available and no candidate remains; not confirmed healthy",
        "pending": "one or more group scores unavailable, or changed input prevents common-pattern review for the affected group/time range",
        "excluded": "existing EXCLUDED rows are preserved; every group label is blank",
        "row_level_or_label_created": False,
        "common_review_source_sha256": REVIEWED_SOURCE_SHA256,
        "common_review_applied": bool(review_applied),
        "common_review_ranges": len(common_ranges),
        "common_review_time_rule": "start inclusive, end exclusive; only the named sensor",
        "common_review_on_changed_input": "no automatic removal; affected group/time ranges pending",
        "complete_lot_statistics": True,
        "future_prediction_evaluation": False,
        "maintenance_used_to_force_label1": False,
        "interpretation_windows_used_to_force_label1": False,
    },
    "retrospective_feature_columns": thermal_input_columns,
    "thermal_feature_policy": {
        "sensors": THERMAL_SENSORS,
        "central_fractions": [CENTRAL_START_FRACTION, CENTRAL_END_FRACTION],
        "reference": "median in middle60% of observed LOT elapsed time, endpoints included, excludes existing exclusions and sensor missing values",
        "standardization": "deviation from central median divided by central60% std ddof0",
        "deviation_threshold_c": TEMP_LIMIT_C,
        "rolling_seconds": ACCUMULATION_SECONDS,
        "max_gap_seconds": MAX_GAP_SECONDS,
        "duration": "elapsed seconds since first consecutive same-direction >5C observation",
        "accumulation": "previous measured excess held until next observation; recent300s integrated excess",
        "coverage_columns": coverage_columns,
        "incomplete_window": "accumulation blank until300s continuous coverage",
        "reference_uses_later_observations": True,
        "allowed_use": "completed LOT retrospective analysis only",
        "included_in_model_input_allowlist": False,
        "labels_changed_by_thermal_features": False,
        "quality_limit_claimed": False,
    },
    "source_sha256": {
        "cal01_base_v1.csv": source_sha256,
        "metadata/v1/dataset.json": input_metadata_sha256,
    },
    "review_scope": "74 reviewed sensor ranges in13 LOTs, 8 domain interpretation windows, existing team-confirmed exclusions; not exhaustive fault confirmation",
    "sampling_applied": False,
    "split_status": "not_assigned",
    "models_trained": False,
    "issue": 99,
}

if SAVE_FILES:
    output.to_csv(OUTPUT_CSV, index=False, encoding=ENC)
    limits.to_csv(os.path.join(REVIEW_DIR, "sigma_limits.csv"), index=False, encoding=ENC)
    common_ranges.to_csv(os.path.join(REVIEW_DIR, "common_pattern_ranges.csv"), index=False, encoding=ENC)
    common_rows.to_csv(os.path.join(REVIEW_DIR, "common_pattern_group_rows.csv"), index=False, encoding=ENC)
    group_counts.to_csv(os.path.join(REVIEW_DIR, "group_label_counts.csv"), index=False, encoding=ENC)
    lot_group_counts.to_csv(os.path.join(REVIEW_DIR, "lot_group_label_counts.csv"), index=False, encoding=ENC)
    domain_counts.to_csv(os.path.join(REVIEW_DIR, "domain_windows.csv"), index=False, encoding=ENC)
    domain_rows.to_csv(os.path.join(REVIEW_DIR, "domain_window_group_rows.csv"), index=False, encoding=ENC)
    thermal_reference.to_csv(os.path.join(REVIEW_DIR, "thermal_reference.csv"), index=False, encoding=ENC)
    thermal_summary.to_csv(os.path.join(REVIEW_DIR, "thermal_summary.csv"), index=False, encoding=ENC)
    with open(OUTPUT_CSV, "rb") as file:
        manifest["master_sha256"] = hashlib.file_digest(file, "sha256").hexdigest()
    # 같은 CSV라도 생성 근거가 달라지면 이전 설정 원문을 따로 보존함
    metadata_text = json.dumps(manifest, ensure_ascii=False, indent=2)
    metadata_sha256 = hashlib.sha256(
        json.dumps(manifest, ensure_ascii=False, sort_keys=True).encode("utf-8")
    ).hexdigest()
    for metadata_name in ["history_" + metadata_sha256 + ".json", manifest["master_sha256"] + ".json", "dataset.json"]:
        metadata_path = os.path.join(META_DIR, metadata_name)
        if metadata_name.startswith("history_") and os.path.isfile(metadata_path):
            continue  # 같은 설정 원문은 이미 저장한 파일을 사용함
        with open(metadata_path, "w", encoding="utf-8", newline="\n") as file:
            file.write(metadata_text)
    print("저장 완료:", OUTPUT_CSV)
else:
    print("저장하지 않고 계산 결과만 확인함")